In [3]:
from pathlib import Path
import subprocess
import pandas as pd
import tempfile

# Load
BASE_DIR = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht")

OPENSMILE_EXE = BASE_DIR / "tools" / "openSMILE" / "bin" / "SMILExtract.exe"
CONFIG_FILE = BASE_DIR / "tools" / "openSMILE" / "config" / "egemaps" / "v02" / "eGeMAPSv02.conf"

perturbation_sets = {
    "gaussian": {
        "input_root": BASE_DIR / "TAME" / "original_gaussian_noise_perturbations_pain",
        "folders": {
            "low_gaussian_noise": "low",
            "medium_gaussian_noise": "medium",
            "high_gaussian_noise": "high",
            "very_high_gaussian_noise": "very_high",
        }
    },
    "lowpass": {
        "input_root": BASE_DIR / "TAME" / "original_lowpass_perturbations_pain",
        "folders": {
            "low_lowpass": "low",
            "medium_lowpass": "medium",
            "high_lowpass": "high",
            "very_high_lowpass": "very_high",
        }
    },
    "pink": {
        "input_root": BASE_DIR / "TAME" / "original_pink_noise_perturbations_pain",
        "folders": {
            "low_pink_noise": "low",
            "medium_pink_noise": "medium",
            "high_pink_noise": "high",
            "very_high_pink_noise": "very_high",
        }
    },
    "intensity": {
        "input_root": BASE_DIR / "TAME" / "original_intensity_perturbations_pain",
        "folders": {
            "-6dB": "-6dB",
            "-3dB": "-3dB",
            "+3dB": "3dB",
            "+6dB": "6dB",
        }
    }
}


# Check main paths
print("SMILExtract exists:", OPENSMILE_EXE.exists())
print("Config exists:", CONFIG_FILE.exists())

if not OPENSMILE_EXE.exists():
    raise FileNotFoundError(f"SMILExtract.exe not found:\n{OPENSMILE_EXE}")

if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Config file not found:\n{CONFIG_FILE}")


def extract_opensmile_features(audio_root, output_features_csv, output_failed_csv):
    print(f"\n--- Processing folder: {audio_root.name} ---")

    if not audio_root.exists():
        raise FileNotFoundError(f"Audio root not found:\n{audio_root}")

    wav_files = sorted(audio_root.rglob("*.wav"))
    print(f"Number of wav files found: {len(wav_files)}")

    if len(wav_files) == 0:
        raise ValueError(f"No .wav files found in:\n{audio_root}")

    all_rows = []
    failed_files = []

    for i, wav_file in enumerate(wav_files, start=1):
        print(f"[{i}/{len(wav_files)}] Processing: {wav_file.name}")

        participant_id = wav_file.parent.name
        filename = wav_file.name

        with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
            temp_output_csv = Path(tmp.name)

        try:
            temp_output_csv.unlink(missing_ok=True)

            cmd = [
                str(OPENSMILE_EXE),
                "-C", str(CONFIG_FILE),
                "-I", str(wav_file),
                "-csvoutput", str(temp_output_csv),
            ]

            result = subprocess.run(cmd, capture_output=True, text=True)

            if result.returncode != 0:
                failed_files.append({
                    "participant_id": participant_id,
                    "filename": filename,
                    "file_path": str(wav_file),
                    "reason": "openSMILE return code != 0",
                    "stderr": result.stderr
                })
                continue

            if not temp_output_csv.exists():
                failed_files.append({
                    "participant_id": participant_id,
                    "filename": filename,
                    "file_path": str(wav_file),
                    "reason": "no output made",
                    "stderr": result.stderr
                })
                continue

            try:
                feature_df = pd.read_csv(temp_output_csv, sep=";", engine="python")
            except Exception as e:
                failed_files.append({
                    "participant_id": participant_id,
                    "filename": filename,
                    "file_path": str(wav_file),
                    "reason": f"error in csv: {e}",
                    "stderr": result.stderr
                })
                continue

            if feature_df.empty:
                failed_files.append({
                    "participant_id": participant_id,
                    "filename": filename,
                    "file_path": str(wav_file),
                    "reason": "output csv is empty",
                    "stderr": result.stderr
                })
                continue

            row = feature_df.iloc[0].to_dict()
            row["name"] = filename
            row["participant_id"] = participant_id
            row["filename"] = filename
            row["file_path"] = str(wav_file)

            all_rows.append(row)

        except Exception as e:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": f"error: {e}",
                "stderr": ""
            })

        finally:
            temp_output_csv.unlink(missing_ok=True)

    features_df = pd.DataFrame(all_rows)

    preferred_front_cols = ["participant_id", "filename", "file_path", "name", "frameTime"]
    remaining_cols = [col for col in features_df.columns if col not in preferred_front_cols]
    features_df = features_df[
        [col for col in preferred_front_cols if col in features_df.columns] + remaining_cols
    ]

    features_df.to_csv(output_features_csv, index=False)

    print("\nDone.")
    print("Number of successful feature rows:", len(features_df))
    print("Number of failed files:", len(failed_files))
    print("Features saved in:")
    print(output_features_csv)

    if failed_files:
        failed_df = pd.DataFrame(failed_files)
        failed_df.to_csv(output_failed_csv, index=False)
        print("\nFailed files saved in:")
        print(output_failed_csv)

    print("\nShape features dataframe:", features_df.shape)
    display(features_df.head())

    return features_df, failed_files


results = {}

for perturbation_type, settings in perturbation_sets.items():
    input_root = settings["input_root"]
    folder_dict = settings["folders"]

    print(f"\n========== {perturbation_type.upper()} ==========")

    if not input_root.exists():
        raise FileNotFoundError(f"Input root not found:\n{input_root}")

    results[perturbation_type] = {}

    for folder_name, label in folder_dict.items():
        audio_root = input_root / folder_name
        output_features_csv = BASE_DIR / f"opensmile_{label}_{perturbation_type}_features_pain_original.csv"
        output_failed_csv = BASE_DIR / f"opensmile_{label}_{perturbation_type}_failed_files_original.csv"

        features_df, failed_files = extract_opensmile_features(
            audio_root=audio_root,
            output_features_csv=output_features_csv,
            output_failed_csv=output_failed_csv
        )

        results[perturbation_type][label] = {
            "features_df": features_df,
            "failed_files": failed_files
        }

SMILExtract exists: True
Config exists: True

========== GAUSSIAN ==========

--- Processing folder: low_gaussian_noise ---
Number of wav files found: 1055
[1/1055] Processing: p10085.LC.1.161.wav
[2/1055] Processing: p10085.LC.12.59.wav
[3/1055] Processing: p10085.LC.14.45.wav
[4/1055] Processing: p10085.LC.18.41.wav
[5/1055] Processing: p10085.LC.2.106.wav
[6/1055] Processing: p10085.LC.23.156.wav
[7/1055] Processing: p10085.LC.24.172.wav
[8/1055] Processing: p10085.LC.26.99999.wav
[9/1055] Processing: p10085.LC.3.60.wav
[10/1055] Processing: p10085.LC.30.132.wav
[11/1055] Processing: p10085.LC.31.99999.wav
[12/1055] Processing: p10085.LC.33.6.wav
[13/1055] Processing: p10085.LC.34.122.wav
[14/1055] Processing: p10085.LC.37.71.wav
[15/1055] Processing: p10085.LC.38.160.wav
[16/1055] Processing: p10085.LC.4.76.wav
[17/1055] Processing: p10085.LC.40.174.wav
[18/1055] Processing: p10085.LC.41.99999.wav
[19/1055] Processing: p10085.LC.5.134.wav
[20/1055] Processing: p10085.LC.6.99999.wav

,participant_id,filename,file_path,name,frameTime,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,...,slopeUV0-500_sma3nz_amean,slopeUV500-1500_sma3nz_amean,spectralFluxUV_sma3nz_amean,loudnessPeaksPerSec,VoicedSegmentsPerSec,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp
0,p10085,p10085.LC.1.161.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.1.161.wav,0.0,35.01972,0.058435,32.85036,34.98619,36.56318,...,-0.021520,-0.012841,0.097341,3.759399,3.065134,0.107500,0.047368,0.175556,0.172118,-34.59005
1,p10085,p10085.LC.12.59.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.12.59.wav,0.0,36.45998,0.054271,35.04662,35.46388,38.76064,...,-0.021852,-0.009274,0.094646,4.245283,2.415459,0.140000,0.077974,0.254000,0.238462,-33.72906
2,p10085,p10085.LC.14.45.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.14.45.wav,0.0,34.32712,0.041923,33.18603,33.72223,35.32508,...,-0.024862,-0.010828,0.097585,3.524229,2.252252,0.196000,0.151737,0.226000,0.256094,-34.10008
3,p10085,p10085.LC.18.41.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.18.41.wav,0.0,34.30899,0.110528,33.53381,34.41047,36.70740,...,-0.014014,-0.017034,0.092545,4.017857,3.196347,0.127143,0.137811,0.165714,0.139577,-35.13419
4,p10085,p10085.LC.2.106.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.2.106.wav,0.0,35.14941,0.062769,33.52361,34.20158,37.36217,...,-0.023645,-0.010545,0.083708,2.966102,2.164502,0.180000,0.093595,0.258000,0.288056,-33.76992



--- Processing folder: medium_gaussian_noise ---
Number of wav files found: 1055
[1/1055] Processing: p10085.LC.1.161.wav
[2/1055] Processing: p10085.LC.12.59.wav
[3/1055] Processing: p10085.LC.14.45.wav
[4/1055] Processing: p10085.LC.18.41.wav
[5/1055] Processing: p10085.LC.2.106.wav
[6/1055] Processing: p10085.LC.23.156.wav
[7/1055] Processing: p10085.LC.24.172.wav
[8/1055] Processing: p10085.LC.26.99999.wav
[9/1055] Processing: p10085.LC.3.60.wav
[10/1055] Processing: p10085.LC.30.132.wav
[11/1055] Processing: p10085.LC.31.99999.wav
[12/1055] Processing: p10085.LC.33.6.wav
[13/1055] Processing: p10085.LC.34.122.wav
[14/1055] Processing: p10085.LC.37.71.wav
[15/1055] Processing: p10085.LC.38.160.wav
[16/1055] Processing: p10085.LC.4.76.wav
[17/1055] Processing: p10085.LC.40.174.wav
[18/1055] Processing: p10085.LC.41.99999.wav
[19/1055] Processing: p10085.LC.5.134.wav
[20/1055] Processing: p10085.LC.6.99999.wav
[21/1055] Processing: p10085.LC.7.62.wav
[22/1055] Processing: p10085.LC.

,participant_id,filename,file_path,name,frameTime,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,...,slopeUV0-500_sma3nz_amean,slopeUV500-1500_sma3nz_amean,spectralFluxUV_sma3nz_amean,loudnessPeaksPerSec,VoicedSegmentsPerSec,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp
0,p10085,p10085.LC.1.161.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.1.161.wav,0.0,35.01963,0.058427,32.85281,34.98716,36.55990,...,-0.021387,-0.012282,0.097348,3.759399,3.065134,0.107500,0.047368,0.175556,0.172118,-34.58901
1,p10085,p10085.LC.12.59.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.12.59.wav,0.0,36.45993,0.054263,35.04636,35.46710,38.76003,...,-0.021831,-0.007673,0.094656,4.245283,2.415459,0.140000,0.077974,0.254000,0.238462,-33.72681
2,p10085,p10085.LC.14.45.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.14.45.wav,0.0,34.32726,0.041923,33.18813,33.72217,35.32865,...,-0.024739,-0.009688,0.097607,3.524229,2.252252,0.196000,0.151737,0.226000,0.256094,-34.09967
3,p10085,p10085.LC.18.41.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.18.41.wav,0.0,34.32080,0.111096,33.58875,34.40994,36.71550,...,-0.014000,-0.015939,0.092258,4.464286,3.196347,0.125714,0.136994,0.167143,0.142900,-35.13229
4,p10085,p10085.LC.2.106.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.2.106.wav,0.0,35.16417,0.062969,33.51703,34.20614,37.38025,...,-0.023814,-0.009232,0.083802,2.966102,2.164502,0.178000,0.093467,0.260000,0.287193,-33.76760



--- Processing folder: high_gaussian_noise ---
Number of wav files found: 1055
[1/1055] Processing: p10085.LC.1.161.wav
[2/1055] Processing: p10085.LC.12.59.wav
[3/1055] Processing: p10085.LC.14.45.wav
[4/1055] Processing: p10085.LC.18.41.wav
[5/1055] Processing: p10085.LC.2.106.wav
[6/1055] Processing: p10085.LC.23.156.wav
[7/1055] Processing: p10085.LC.24.172.wav
[8/1055] Processing: p10085.LC.26.99999.wav
[9/1055] Processing: p10085.LC.3.60.wav
[10/1055] Processing: p10085.LC.30.132.wav
[11/1055] Processing: p10085.LC.31.99999.wav
[12/1055] Processing: p10085.LC.33.6.wav
[13/1055] Processing: p10085.LC.34.122.wav
[14/1055] Processing: p10085.LC.37.71.wav
[15/1055] Processing: p10085.LC.38.160.wav
[16/1055] Processing: p10085.LC.4.76.wav
[17/1055] Processing: p10085.LC.40.174.wav
[18/1055] Processing: p10085.LC.41.99999.wav
[19/1055] Processing: p10085.LC.5.134.wav
[20/1055] Processing: p10085.LC.6.99999.wav
[21/1055] Processing: p10085.LC.7.62.wav
[22/1055] Processing: p10085.LC.9.

...

,participant_id,filename,file_path,name,frameTime,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,...,slopeUV0-500_sma3nz_amean,slopeUV500-1500_sma3nz_amean,spectralFluxUV_sma3nz_amean,loudnessPeaksPerSec,VoicedSegmentsPerSec,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp
0,p10085,p10085.LC.1.161.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.1.161.wav,0.0,35.01982,0.058435,32.85137,34.98691,36.56270,...,-0.019098,-0.011328,0.068892,3.759399,3.065134,0.107500,0.047368,0.175556,0.172118,-37.59200
1,p10085,p10085.LC.12.59.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.12.59.wav,0.0,36.45982,0.054269,35.04666,35.46404,38.76054,...,-0.019442,-0.007535,0.066964,4.245283,2.415459,0.140000,0.077974,0.254000,0.238462,-36.73072
2,p10085,p10085.LC.14.45.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.14.45.wav,0.0,34.32711,0.041927,33.18596,33.72219,35.32539,...,-0.022452,-0.009344,0.069053,3.524229,2.252252,0.196000,0.151737,0.226000,0.256094,-37.10236
3,p10085,p10085.LC.18.41.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.18.41.wav,0.0,34.30878,0.110532,33.53381,34.41002,36.70700,...,-0.011595,-0.015316,0.065491,4.017857,3.196347,0.127143,0.137811,0.165714,0.139577,-38.13634
4,p10085,p10085.LC.2.106.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.2.106.wav,0.0,35.14958,0.062773,33.52359,34.20160,37.36095,...,-0.021253,-0.009010,0.059224,2.966102,2.164502,0.180000,0.093595,0.258000,0.288056,-36.77179



--- Processing folder: +3dB ---
Number of wav files found: 1055
[1/1055] Processing: p10085.LC.1.161.wav
[2/1055] Processing: p10085.LC.12.59.wav
[3/1055] Processing: p10085.LC.14.45.wav
[4/1055] Processing: p10085.LC.18.41.wav
[5/1055] Processing: p10085.LC.2.106.wav
[6/1055] Processing: p10085.LC.23.156.wav
[7/1055] Processing: p10085.LC.24.172.wav
[8/1055] Processing: p10085.LC.26.99999.wav
[9/1055] Processing: p10085.LC.3.60.wav
[10/1055] Processing: p10085.LC.30.132.wav
[11/1055] Processing: p10085.LC.31.99999.wav
[12/1055] Processing: p10085.LC.33.6.wav
[13/1055] Processing: p10085.LC.34.122.wav
[14/1055] Processing: p10085.LC.37.71.wav
[15/1055] Processing: p10085.LC.38.160.wav
[16/1055] Processing: p10085.LC.4.76.wav
[17/1055] Processing: p10085.LC.40.174.wav
[18/1055] Processing: p10085.LC.41.99999.wav
[19/1055] Processing: p10085.LC.5.134.wav
[20/1055] Processing: p10085.LC.6.99999.wav
[21/1055] Processing: p10085.LC.7.62.wav
[22/1055] Processing: p10085.LC.9.55.wav
[23/1055

,participant_id,filename,file_path,name,frameTime,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,...,slopeUV0-500_sma3nz_amean,slopeUV500-1500_sma3nz_amean,spectralFluxUV_sma3nz_amean,loudnessPeaksPerSec,VoicedSegmentsPerSec,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp
0,p10085,p10085.LC.1.161.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.1.161.wav,0.0,35.01982,0.058435,32.85135,34.98661,36.56275,...,-0.023911,-0.014601,0.137553,3.759399,3.065134,0.107500,0.047368,0.175556,0.172118,-31.58870
1,p10085,p10085.LC.12.59.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.12.59.wav,0.0,36.45980,0.054268,35.04674,35.46412,38.76046,...,-0.024243,-0.010789,0.133698,4.245283,2.415459,0.140000,0.077974,0.254000,0.238462,-30.72772
2,p10085,p10085.LC.14.45.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.14.45.wav,0.0,34.32709,0.041926,33.18596,33.72220,35.32523,...,-0.027261,-0.012615,0.137877,3.524229,2.252252,0.196000,0.151737,0.226000,0.256094,-31.09913
3,p10085,p10085.LC.18.41.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.18.41.wav,0.0,34.30872,0.110542,33.53382,34.41000,36.70711,...,-0.016414,-0.018602,0.130766,4.017857,3.196347,0.127143,0.137811,0.165714,0.139577,-32.13254
4,p10085,p10085.LC.2.106.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.2.106.wav,0.0,35.14965,0.062775,33.52358,34.20151,37.36132,...,-0.026046,-0.012304,0.118262,2.966102,2.164502,0.180000,0.093595,0.258000,0.288056,-30.76879



--- Processing folder: +6dB ---
Number of wav files found: 1055
[1/1055] Processing: p10085.LC.1.161.wav
[2/1055] Processing: p10085.LC.12.59.wav
[3/1055] Processing: p10085.LC.14.45.wav
[4/1055] Processing: p10085.LC.18.41.wav
[5/1055] Processing: p10085.LC.2.106.wav
[6/1055] Processing: p10085.LC.23.156.wav
[7/1055] Processing: p10085.LC.24.172.wav
[8/1055] Processing: p10085.LC.26.99999.wav
[9/1055] Processing: p10085.LC.3.60.wav
[10/1055] Processing: p10085.LC.30.132.wav
[11/1055] Processing: p10085.LC.31.99999.wav
[12/1055] Processing: p10085.LC.33.6.wav
[13/1055] Processing: p10085.LC.34.122.wav
[14/1055] Processing: p10085.LC.37.71.wav
[15/1055] Processing: p10085.LC.38.160.wav
[16/1055] Processing: p10085.LC.4.76.wav
[17/1055] Processing: p10085.LC.40.174.wav
[18/1055] Processing: p10085.LC.41.99999.wav
[19/1055] Processing: p10085.LC.5.134.wav
[20/1055] Processing: p10085.LC.6.99999.wav
[21/1055] Processing: p10085.LC.7.62.wav
[22/1055] Processing: p10085.LC.9.55.wav
[23/1055

,participant_id,filename,file_path,name,frameTime,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,...,slopeUV0-500_sma3nz_amean,slopeUV500-1500_sma3nz_amean,spectralFluxUV_sma3nz_amean,loudnessPeaksPerSec,VoicedSegmentsPerSec,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp
0,p10085,p10085.LC.1.161.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.1.161.wav,0.0,35.01984,0.058435,32.85135,34.98690,36.56286,...,-0.026313,-0.016247,0.194337,3.759399,3.065134,0.107500,0.047368,0.175556,0.172118,-28.58768
1,p10085,p10085.LC.12.59.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.12.59.wav,0.0,36.45980,0.054268,35.04674,35.46410,38.76045,...,-0.026646,-0.012436,0.188892,4.245283,2.415459,0.140000,0.077974,0.254000,0.238462,-27.72683
2,p10085,p10085.LC.14.45.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.14.45.wav,0.0,34.32713,0.041927,33.18588,33.72220,35.32593,...,-0.029667,-0.014263,0.194797,3.524229,2.252252,0.196000,0.151737,0.226000,0.256094,-28.09820
3,p10085,p10085.LC.18.41.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.18.41.wav,0.0,34.30873,0.110540,33.53381,34.41000,36.70712,...,-0.018814,-0.020257,0.184749,4.017857,3.196347,0.127143,0.137811,0.165714,0.139577,-29.13141
4,p10085,p10085.LC.2.106.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.2.106.wav,0.0,35.14960,0.062773,33.52357,34.20152,37.36135,...,-0.028444,-0.013930,0.167088,2.966102,2.164502,0.180000,0.093595,0.258000,0.288056,-27.76791
